In [3]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import uproot
from datetime import datetime
import plotly.graph_objects as go
import re
import awkward as ak
import pandas as pd


In [4]:
NEW_DATA_FILE = '/main/run3_datagen/data/MuonSystem_Tree.root'
OLD_DATA_FILE = '/main/run3_datagen/data/samples/displacedJetMuon_ntupler_10.root'
OUTPUT_DIR = '/main/run3_datagen/notebooks/model_compatibility/images'

old_tree_file = uproot.open(OLD_DATA_FILE)
old_tree = old_tree_file['ntuples/llp']

new_tree_file = uproot.open(NEW_DATA_FILE)
new_tree = new_tree_file['MuonSystem']

old_tree_keys = old_tree.keys()
new_tree_keys = new_tree.keys()


In [5]:
def load_data(tree, var_name):
    """Load data from tree"""
    return tree[var_name].array(library="ak")


In [ ]:
# Helper function to normalize keys: lowercase and strip special characters
def normalize_keys(keys):
    return {re.sub(r'\W+', '', key.lower()): key for key in keys}

# Normalize the keys
old_keys_normalized = normalize_keys(old_tree_keys)
new_keys_normalized = normalize_keys(new_tree_keys)

# Find intersection and differences
common_keys = set(old_keys_normalized.keys()) & set(new_keys_normalized.keys())
only_in_old = set(old_keys_normalized.keys()) - set(new_keys_normalized.keys())
only_in_new = set(new_keys_normalized.keys()) - set(old_keys_normalized.keys())


: 

In [ ]:
for norm_key in sorted(common_keys):
    var_old = old_keys_normalized[norm_key]
    var_new = new_keys_normalized[norm_key]
    filename = f"{norm_key}.png"
    filepath = f"{OUTPUT_DIR}/{norm_key}.png"

    try:
        # Load and flatten arrays
        old_data_raw = load_data(old_tree, var_old)
        new_data_raw = load_data(new_tree, var_new)
        
        old_array = ak.flatten(old_data_raw, axis=None)
        new_array = ak.flatten(new_data_raw, axis=None)

        # Convert to numpy
        old_data = ak.to_numpy(old_array)
        new_data = ak.to_numpy(new_array)

        # Ensure numeric types
        if not np.issubdtype(old_data.dtype, np.number) or not np.issubdtype(new_data.dtype, np.number):
            raise ValueError("Non-numeric data type")

        # Remove NaNs and infs
        old_data = old_data[np.isfinite(old_data)]
        new_data = new_data[np.isfinite(new_data)]

        # Binning
        combined = np.concatenate((old_data, new_data))
        bins = np.histogram_bin_edges(combined, bins="auto")

        # New: GluGluH-Hto2Sto4B_Par-ctauS-1000-MH-125-MS-15_TuneCP5_13p6TeV_powheg-pythia8
        # OLD: ggH_Hto2Sto4B_MH-125-MS-15-ctauS-1000_TuneCP5_13p6TeV_powheg-pythia8

        # Plot
        plt.figure(figsize=(8, 5))
        plt.hist(old_data, bins=bins, alpha=0.5, label=f"OLD Events", color="blue", density=True)
        plt.hist(new_data, bins=bins, alpha=0.5, label=f"NEW Events", color="red", density=True)
        plt.title(f"Histogram: {var_old}")
        plt.xlabel(var_old)
        plt.ylabel("Density")
        plt.legend()
        plt.savefig(filepath)
        plt.close()

    except Exception as e:
        # Plot blank image with explanation
        plt.figure(figsize=(8, 5))
        plt.text(0.5, 0.5, f"{var_old}\n\nUnable to plot\n({str(e)})",
                 fontsize=12, ha='center', va='center', wrap=True)
        plt.axis('off')
        plt.title(f"{var_old} - Skipped")
        plt.savefig(filepath)
        plt.close()
        print(f"Blank saved for {var_old}: {e}")

print(f"\nHistograms (and blanks) saved to: {OUTPUT_DIR}")


/home/admin/miniforge3/envs/mdsml/lib/python3.10/site-packages/numpy/lib/_histograms_impl.py:901: RuntimeWarning: invalid value encountered in divide
  return n/db/n.sum(), bin_edges


Blank saved for Flag_BadPFMuonFilter: Non-numeric data type
Blank saved for Flag_ecalBadCalibFilter: Non-numeric data type
Blank saved for Flag_EcalDeadCellTriggerPrimitiveFilter: Non-numeric data type
Blank saved for Flag_eeBadScFilter: Non-numeric data type
Blank saved for Flag_globalSuperTightHalo2016Filter: Non-numeric data type
Blank saved for Flag_goodVertices: Non-numeric data type


/tmp/ipykernel_2680/2675503729.py:42: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(filepath)


Blank saved for gLLP_csc: Non-numeric data type
Blank saved for gLLP_dt: Non-numeric data type
